In [1]:

# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spaceship-titanic/sample_submission.csv
/kaggle/input/spaceship-titanic/train.csv
/kaggle/input/spaceship-titanic/test.csv


In [2]:

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ML imports
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except Exception:
    LIGHTGBM_AVAILABLE = False

print("LightGBM available:", LIGHTGBM_AVAILABLE)

# -------------------------
# 1) Load dataset
# -------------------------
train = pd.read_csv("/kaggle/input/spaceship-titanic/train.csv")
test  = pd.read_csv("/kaggle/input/spaceship-titanic/test.csv")
test_passenger_ids = test["PassengerId"].copy()

# -------------------------
# 2) Preprocessing & feature engineering
# -------------------------
def basic_fe(df):
    df = df.copy()
    # Cabin -> Deck / Room / Side
    cab = df["Cabin"].astype(str).fillna("Unknown/0/Unknown")
    parts = cab.str.split("/", expand=True)
    df["Deck"] = parts[0].fillna("Unknown")
    df["Room"] = parts[1].fillna("0")
    df["Side"] = parts[2].fillna("Unknown")
    # Room numeric & parity
    df["RoomNumber"] = pd.to_numeric(df["Room"], errors="coerce").fillna(0).astype(int)
    df["RoomEven"] = (df["RoomNumber"] % 2 == 0).astype(str)
    # GroupID and GroupSize (PassengerId format like 0001_01)
    df["GroupID"] = df["PassengerId"].astype(str).str.split("_").str[0]
    df["GroupSize"] = df["GroupID"].map(df["GroupID"].value_counts()).fillna(1).astype(int)
    df["IsAlone"] = (df["GroupSize"] == 1).astype(str)
    # TotalSpent and log
    money_cols = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]
    for m in money_cols:
        if m in df.columns:
            df[m] = pd.to_numeric(df[m], errors="coerce").fillna(0.0)
    df["TotalSpent"] = df[money_cols].sum(axis=1)
    df["LogTotalSpent"] = np.log1p(df["TotalSpent"])
    # Extract Title from Name (e.g. "Braund, Mr. Owen Harris" -> "Mr")
    df["Title"] = df["Name"].astype(str).str.extract(r',\s*([^\.]+)\.', expand=False).fillna("Unknown")
    # Convert boolean-like columns that may be actual booleans to string for tree models
    for c in ["VIP","CryoSleep"]:
        if c in df.columns:
            # Some datasets have booleans or strings -- normalize to string
            df[c] = df[c].fillna(False).astype(str)
    # Fill HomePlanet/Destination
    for c in ["HomePlanet","Destination"]:
        if c in df.columns:
            df[c] = df[c].fillna("Unknown").astype(str)
    return df

train_fe = basic_fe(train)
test_fe  = basic_fe(test)

# Fill Age missing with median from train
age_median = train_fe["Age"].median()
train_fe["Age"] = train_fe["Age"].fillna(age_median)
test_fe["Age"]  = test_fe["Age"].fillna(age_median)

# Choose features to include
drop_cols = ["PassengerId", "Cabin", "Name", "GroupID", "Room"]  # raws we don't pass directly
# We'll keep the engineered columns and numeric money fields
candidate_features = [
    "HomePlanet", "CryoSleep", "Destination", "Age", "VIP",
    "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
    "TotalSpent", "LogTotalSpent", "Deck", "RoomNumber", "RoomEven",
    "Side", "GroupSize", "IsAlone", "Title"
]

# Ensure columns exist
candidate_features = [c for c in candidate_features if c in train_fe.columns]

train_input = train_fe.drop(columns=[c for c in drop_cols if c in train_fe.columns]).copy()
test_input  = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns]).copy()

# Make sure target is numeric 0/1
train_input["Transported"] = train_input["Transported"].astype(int)

# -------------
# 3) Encoding categoricals for tree models
# -------------
# For tree-based models, ordinal encoding of categories is fine. Fit on combined data so unseen categories map to -1.
cat_cols = [c for c in candidate_features if train_input[c].dtype == "object" or train_input[c].dtype == "bool" or isinstance(train_input[c].dtype, pd.CategoricalDtype)]
# Convert everything to str for categorical columns (safe)
for c in cat_cols:
    train_input[c] = train_input[c].astype(str)
    test_input[c]  = test_input[c].astype(str)

# Fit OrdinalEncoder on combined categories
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
if len(cat_cols) > 0:
    encoder.fit(pd.concat([train_input[cat_cols], test_input[cat_cols]], axis=0))
    train_cat_enc = encoder.transform(train_input[cat_cols])
    test_cat_enc  = encoder.transform(test_input[cat_cols])
    # Replace columns
    train_enc = train_input.copy()
    test_enc  = test_input.copy()
    for i, c in enumerate(cat_cols):
        train_enc[c] = train_cat_enc[:, i].astype(int)
        test_enc[c]  = test_cat_enc[:, i].astype(int)
else:
    train_enc = train_input.copy()
    test_enc = test_input.copy()

# Final features list
features = [c for c in candidate_features if c in train_enc.columns]
print("Using features:", features)
print("Train shape:", train_enc.shape, "Test shape:", test_enc.shape)

# Build X and y
X = train_enc[features].copy()
y = train_enc["Transported"].copy()
X_test = test_enc[features].copy()

# Simple imputer for any remaining numeric NAs
num_imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(num_imputer.fit_transform(X), columns=X.columns, index=X.index)
X_test = pd.DataFrame(num_imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# -------------------------
# 4) Prepare base models
# -------------------------
base_models = []
base_models.append(("rf", RandomForestClassifier(n_estimators=300, max_depth=None, n_jobs=-1, random_state=42)))
base_models.append(("hgb", HistGradientBoostingClassifier(random_state=42)))

if LIGHTGBM_AVAILABLE:
    base_models.append(("lgbm", lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05,
                                                   num_leaves=31, random_state=42)))

print("Base models:", [name for name,_ in base_models])

# -------------------------
# 5) Out-of-fold stacking generation
# -------------------------
n_models = len(base_models)
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

S_train = np.zeros((X.shape[0], n_models))
S_test = np.zeros((X_test.shape[0], n_models))

model_fold_scores = {name: [] for name, _ in base_models}

for i, (name, model) in enumerate(base_models):
    print(f"\n>>>> Training base model {i+1}/{n_models}: {name}")
    test_fold_preds = np.zeros((n_splits, X_test.shape[0]))
    oof_preds = np.zeros(X.shape[0])
    fold_idx = 0
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        # Fit
        model.fit(X_tr, y_tr)
        # Predict proba for validation
        if hasattr(model, "predict_proba"):
            val_pred = model.predict_proba(X_val)[:, 1]
            test_pred = model.predict_proba(X_test)[:, 1]
        else:
            # Some estimators (rare) may not implement predict_proba
            val_pred = model.predict(X_val)
            test_pred = model.predict(X_test)
        oof_preds[val_idx] = val_pred
        test_fold_preds[fold_idx, :] = test_pred
        # fold metric
        fold_acc = accuracy_score(y_val, (val_pred > 0.5).astype(int))
        model_fold_scores[name].append(fold_acc)
        print(f"  fold {fold_idx+1} acc: {fold_acc:.4f}")
        fold_idx += 1
    # Save OOF column and average test predictions
    S_train[:, i] = oof_preds
    S_test[:, i] = test_fold_preds.mean(axis=0)
    print(f"{name} CV acc mean: {np.mean(model_fold_scores[name]):.4f} std: {np.std(model_fold_scores[name]):.4f}")

# Quick OOF baseline check: if any base model OOF column is useful, its correlation with y should be > 0
for i, (name, _) in enumerate(base_models):
    oof_binary = (S_train[:, i] > 0.5).astype(int)
    print(f"OOF acc for {name}: {accuracy_score(y, oof_binary):.4f}  ROC-AUC: {roc_auc_score(y, S_train[:, i]):.4f}")

# -------------------------
# 6) Fit meta-model (Logistic Regression) on OOF features
# -------------------------
meta = LogisticRegression(max_iter=2000, solver="lbfgs")
meta.fit(S_train, y)
# Evaluate meta using simple hold-out from OOF (we can do kfold CV here as well)
meta_oof_pred = meta.predict(S_train)
print("\nMeta (stack) accuracy on OOF training features:", accuracy_score(y, meta_oof_pred))
if hasattr(meta, "predict_proba"):
    print("Meta ROC-AUC on OOF:", roc_auc_score(y, meta.predict_proba(S_train)[:,1]))

# -------------------------
# 7) Final predictions and submission
# -------------------------
final_probs = meta.predict_proba(S_test)[:, 1] if hasattr(meta, "predict_proba") else meta.predict(S_test)
final_labels = (final_probs > 0.5).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Transported": final_labels
})

out_path = "/kaggle/working/submission.csv"
submission.to_csv(out_path, index=False)
print(f"\nSaved submission -> {out_path}")
print("First rows:")
print(submission.head())

# Verify file exists and list working dir
print("\nFiles in /kaggle/working/:")
print(os.listdir("/kaggle/working"))

# Optional: show simple variable importances if LightGBM used
if LIGHTGBM_AVAILABLE:
    print("\nLightGBM feature importances (gain):")
    lgb_model = [m for n,m in base_models if n=="lgbm"][0]
    try:
        fi = pd.DataFrame({
            "feature": features,
            "importance": lgb_model.feature_importances_
        }).sort_values("importance", ascending=False)
        print(fi.head(20))
    except Exception as e:
        print("Couldn't show LGB feature importances:", e)

print("\nDone.")


LightGBM available: True
Using features: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpent', 'LogTotalSpent', 'Deck', 'RoomNumber', 'RoomEven', 'Side', 'GroupSize', 'IsAlone', 'Title']
Train shape: (8693, 20) Test shape: (4277, 19)
Base models: ['rf', 'hgb', 'lgbm']

>>>> Training base model 1/3: rf
  fold 1 acc: 0.8194
  fold 2 acc: 0.7947
  fold 3 acc: 0.8005
  fold 4 acc: 0.7946
  fold 5 acc: 0.7848
rf CV acc mean: 0.7988 std: 0.0115

>>>> Training base model 2/3: hgb
  fold 1 acc: 0.8125
  fold 2 acc: 0.8143
  fold 3 acc: 0.8125
  fold 4 acc: 0.8165
  fold 5 acc: 0.8072
hgb CV acc mean: 0.8126 std: 0.0030

>>>> Training base model 3/3: lgbm
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002865 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can